# Llama-3-8B Cross-Linguistic Consistency Check

The critical test: does the within-probe exponent consistency we saw with Mistral
replicate with an independent 8B model?

**Mistral within-probe**: 8 languages, mean α=-0.71, SD=0.08
**Llama RAID reference**: human α=-1.26, AI α=-1.51

If Llama shows tight clustering across languages (even at a different absolute value),
the cross-linguistic consistency is real. If it shows random scatter, the Mistral
result was model-specific.

Running on 6 Wikipedia languages + Buckeye + French = 8 datasets.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import uniform_filter1d
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print('Imports OK')

In [ ]:
# === Configuration ===
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_BASE = Path('/content/drive/MyDrive/LRTIA/Data')
    BASE_DIR = Path('/content/drive/MyDrive/LRTIA/Results/Llama_crosslingual')
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    DATA_BASE = Path('../data')
    BASE_DIR = Path('../results/Llama_crosslingual')
    BASE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'
USE_4BIT = True
MAX_CONTEXT = 100
TARGET_LEN = 30
TARGET_FRACTIONS = [0.25, 0.50, 0.75]
MIN_CONTEXT_BEFORE_TARGET = MAX_CONTEXT + 10
RANDOM_SEED = 42

# All datasets
DATASETS = {}

lang_names = {'zh': 'Chinese', 'ja': 'Japanese', 'ko': 'Korean',
              'tr': 'Turkish', 'ar': 'Arabic', 'fi': 'Finnish'}
for lang, name in lang_names.items():
    p = DATA_BASE / 'wiki_multilingual' / f'{lang}_articles.jsonl'
    if p.exists():
        DATASETS[f'wiki_{lang}'] = {'path': p, 'label': f'{name} Wiki'}

bk = DATA_BASE / 'buckeye_processed' / 'speaker_concatenated.jsonl'
if bk.exists():
    DATASETS['buckeye'] = {'path': bk, 'label': 'Buckeye spoken'}

fr = DATA_BASE / 'french_oral_processed' / 'per_story.jsonl'
if fr.exists():
    DATASETS['french'] = {'path': fr, 'label': 'French spoken'}

print(f'Datasets found: {len(DATASETS)}')
for k, v in DATASETS.items():
    print(f'  {v["label"]}')

In [ ]:
# === Load Llama-3-8B (4-bit) ===
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto'
)
model.eval()
print('Model loaded')

In [ ]:
# === Core functions ===
common_x = np.arange(1, MAX_CONTEXT + 1)
bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

@torch.no_grad()
def compute_ppl(token_ids, target_start, target_end):
    if target_start >= target_end - 1:
        return float('inf')
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[token_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')

def compute_token_reveal_curve(full_ids, target_start, target_end,
                                shuffled=False, rng_shuf=None):
    target_ids = full_ids[target_start:target_end]
    context_pool = list(full_ids[:target_start])
    if shuffled and rng_shuf is not None:
        context_pool = list(context_pool)
        rng_shuf.shuffle(context_pool)
    max_ctx = min(MAX_CONTEXT, len(context_pool))
    if max_ctx < 10:
        return None
    ppls, ctx_lengths = [], []
    for ctx_len in range(1, max_ctx + 1):
        ctx_tokens = context_pool[-ctx_len:]
        chunk = ctx_tokens + target_ids
        ppl = compute_ppl(chunk, len(ctx_tokens), len(chunk))
        if not math.isinf(ppl):
            ppls.append(ppl)
            ctx_lengths.append(ctx_len)
    if len(ppls) < 10:
        return None
    return {'ctx_lengths': ctx_lengths, 'ppls': ppls}

def process_document(doc, rng_shuf):
    full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
    n = len(full_ids)
    intact_curves, shuffled_curves = [], []
    for frac in TARGET_FRACTIONS:
        target_start = int(n * frac)
        target_end = min(target_start + TARGET_LEN, n)
        if target_start < MIN_CONTEXT_BEFORE_TARGET or target_end - target_start < 5:
            continue
        result = compute_token_reveal_curve(full_ids, target_start, target_end)
        if result is not None:
            result['doc_id'] = doc.get('doc_id', '')
            result['target_frac'] = frac
            intact_curves.append(result)
        result_s = compute_token_reveal_curve(full_ids, target_start, target_end,
                                              shuffled=True, rng_shuf=rng_shuf)
        if result_s is not None:
            result_s['doc_id'] = doc.get('doc_id', '')
            result_s['target_frac'] = frac
            shuffled_curves.append(result_s)
    return intact_curves, shuffled_curves

def compute_raw_ppl_curve(curves):
    all_ppl = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        interp = np.interp(common_x, ctx, ppl, left=np.nan, right=np.nan)
        all_ppl.append(interp)
    return np.nanmean(np.array(all_ppl), axis=0)

def fit_power_law(marg):
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        slope, intercept, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return slope, r, p, bc, bm, intercept
    return None

print('Functions defined')

In [ ]:
# === Process all datasets ===
all_results = {}

for key, info in DATASETS.items():
    print(f'\n{"="*60}')
    print(f'{info["label"]} ({key})')
    print(f'{"="*60}')

    intact_path = BASE_DIR / f'{key}_intact.json'
    shuffled_path = BASE_DIR / f'{key}_shuffled.json'

    if intact_path.exists() and shuffled_path.exists():
        with open(intact_path) as f:
            intact = json.load(f)
        with open(shuffled_path) as f:
            shuffled = json.load(f)
        print(f'  Loaded {len(intact)} intact + {len(shuffled)} shuffled from cache')
    else:
        corpus = []
        with open(info['path']) as f:
            for line in f:
                corpus.append(json.loads(line))
        print(f'  Loaded {len(corpus)} documents')

        intact, shuffled = [], []
        rng_shuf = np.random.RandomState(RANDOM_SEED + 99)
        for doc in tqdm(corpus, desc=info['label']):
            i, s = process_document(doc, rng_shuf)
            intact.extend(i)
            shuffled.extend(s)

        with open(intact_path, 'w') as f:
            json.dump(intact, f)
        with open(shuffled_path, 'w') as f:
            json.dump(shuffled, f)
        print(f'  Computed {len(intact)} intact + {len(shuffled)} shuffled')

    if len(intact) >= 5 and len(shuffled) >= 5:
        ip = compute_raw_ppl_curve(intact)
        sp = compute_raw_ppl_curve(shuffled)
        corr = -np.diff(ip) - (-np.diff(sp))
        fit = fit_power_law(corr)
        if fit:
            all_results[key] = {
                'label': info['label'], 'slope': fit[0], 'r': fit[1], 'p': fit[2],
                'n': len(intact), 'corrected_marg': corr,
                'bc': fit[3], 'bm': fit[4], 'intercept': fit[5],
            }
            print(f'  >> α = {fit[0]:.3f} (r={fit[1]:.3f}, p={fit[2]:.4f})')
        else:
            print(f'  >> Fit failed')
    else:
        print(f'  >> Not enough curves')

print(f'\n\nDone! {len(all_results)} datasets with fits')

In [ ]:
# === Summary + Comparison with Mistral ===
LLAMA_RAID_REF = {'label': 'RAID English (ref)', 'slope': -1.264, 'r': -0.963}

# Mistral results for comparison
MISTRAL = {
    'RAID English': -0.75,
    'Buckeye spoken': -0.73,
    'French spoken': -0.69,
    'Chinese Wiki': -0.82,
    'Japanese Wiki': -0.72,
    'Korean Wiki': -0.69,
    'Turkish Wiki': -0.79,
    'Arabic Wiki': -0.61,
    'Finnish Wiki': -0.56,
}

print(f'{"Dataset":<25} {"Llama α":>10} {"Llama r":>10} {"Mistral α":>12}')
print('-' * 60)
print(f'{LLAMA_RAID_REF["label"]:<25} {LLAMA_RAID_REF["slope"]:>10.3f} {LLAMA_RAID_REF["r"]:>10.3f} {MISTRAL.get("RAID English", 0):>12.3f}')
for key, res in all_results.items():
    m_val = MISTRAL.get(res['label'], None)
    m_str = f'{m_val:.3f}' if m_val else '—'
    print(f'{res["label"]:<25} {res["slope"]:>10.3f} {res["r"]:>10.3f} {m_str:>12}')

# Llama stats
llama_exps = [LLAMA_RAID_REF['slope']] + [r['slope'] for r in all_results.values()]
llama_strong = [LLAMA_RAID_REF['slope']] + [r['slope'] for r in all_results.values() if r['r'] < -0.80]
mistral_exps = list(MISTRAL.values())

print(f'\nLlama (all):    mean={np.mean(llama_exps):.3f}, SD={np.std(llama_exps):.3f}')
if len(llama_strong) > 1:
    print(f'Llama (r<-0.8): mean={np.mean(llama_strong):.3f}, SD={np.std(llama_strong):.3f}')
print(f'Mistral (all):  mean={np.mean(mistral_exps):.3f}, SD={np.std(mistral_exps):.3f}')

# Correlation between Llama and Mistral exponents (paired by dataset)
paired_l, paired_m = [], []
for key, res in all_results.items():
    m_val = MISTRAL.get(res['label'], None)
    if m_val is not None:
        paired_l.append(res['slope'])
        paired_m.append(m_val)
if len(paired_l) >= 3:
    r_corr, p_corr = stats.pearsonr(paired_l, paired_m)
    print(f'\nLlama-Mistral correlation across datasets: r={r_corr:.3f}, p={p_corr:.4f}')
    print('  (If r is high, both probes see the same relative pattern across languages)')

In [ ]:
# === Figure ===
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel A: Llama exponents
ax = axes[0]
labels = [LLAMA_RAID_REF['label']] + [r['label'] for r in all_results.values()]
exponents = [LLAMA_RAID_REF['slope']] + [r['slope'] for r in all_results.values()]
ax.bar(range(len(labels)), exponents, color='#9467bd', alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
ax.set_ylabel('Exponent (α)')
ax.set_title('Llama-3-8B Across Languages', fontweight='bold')
ax.axhline(np.mean(exponents), color='black', linestyle='-', linewidth=2, alpha=0.5,
           label=f'Mean: {np.mean(exponents):.2f}')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis='y')
for i, exp in enumerate(exponents):
    ax.text(i, exp - 0.05, f'{exp:.2f}', ha='center', fontsize=8, fontweight='bold')

# Panel B: Llama vs Mistral paired comparison
ax = axes[1]
if len(paired_l) >= 3:
    ax.scatter(paired_m, paired_l, s=80, c='#9467bd', edgecolors='black', zorder=5)
    for i, key in enumerate([k for k in all_results if MISTRAL.get(all_results[k]['label'])]):
        ax.annotate(all_results[key]['label'], (paired_m[i], paired_l[i]),
                    fontsize=7, ha='left', va='bottom', xytext=(5, 5),
                    textcoords='offset points')
    # Fit line
    z = np.polyfit(paired_m, paired_l, 1)
    x_line = np.linspace(min(paired_m) - 0.05, max(paired_m) + 0.05, 100)
    ax.plot(x_line, np.polyval(z, x_line), '--', color='gray', alpha=0.5)
    ax.set_xlabel('Mistral exponent')
    ax.set_ylabel('Llama exponent')
    ax.set_title(f'Llama vs Mistral (r={r_corr:.2f}, p={p_corr:.3f})', fontweight='bold')
    ax.grid(True, alpha=0.2)
else:
    ax.text(0.5, 0.5, 'Not enough paired data', transform=ax.transAxes, ha='center')

# Panel C: Both probes side by side
ax = axes[2]
common_labels = []
m_vals, l_vals = [], []
for key, res in all_results.items():
    m_val = MISTRAL.get(res['label'])
    if m_val is not None:
        common_labels.append(res['label'])
        m_vals.append(m_val)
        l_vals.append(res['slope'])
x_pos = np.arange(len(common_labels))
w = 0.35
ax.bar(x_pos - w/2, m_vals, w, color='#1f77b4', alpha=0.7, label='Mistral', edgecolor='black')
ax.bar(x_pos + w/2, l_vals, w, color='#9467bd', alpha=0.7, label='Llama', edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels(common_labels, fontsize=8, rotation=45, ha='right')
ax.set_ylabel('Exponent (α)')
ax.set_title('Mistral vs Llama: Same Languages', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis='y')

plt.suptitle('Cross-Probe Replication: Llama-3-8B vs Mistral-7B',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_llama_crosslingual.png', dpi=150, bbox_inches='tight')
plt.show()

## Verdict

Three possible outcomes:

1. **Llama exponents cluster tightly AND correlate with Mistral** → both probes see the same
   cross-linguistic pattern, the finding is real
2. **Llama exponents cluster tightly but DON'T correlate with Mistral** → each probe measures
   something consistent but different, need to understand what
3. **Llama exponents scatter randomly** → the Mistral clustering was model-specific, finding is weak